In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-large',
    # model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_KEY"),
    base_url="https://api.openai.com/v1"
)

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.3,
    base_url="https://api.openai.com/v1",
    api_key=os.getenv("OPENAI_KEY")
)

INPUT_FILE = "../inputs/yaskawa-l1000a.txt"
DB_DIR = "../ChromaDB/db"

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:

from libAgent.markdownSplitter import markdownTextSplitter
chunks = markdownTextSplitter(INPUT_FILE)

from libAgent.retriver import RetriverFactory

# Dense Retriever (MMR)
dense_retriever = RetriverFactory.createChromaRetriverMMR(embeddings=embeddings,dbPath=DB_DIR)

# Sparse Retriever (BM25)
bm25_retriever = RetriverFactory.createBM25RetrieverFromDocuments(chunks)


# Hybrid Retriever
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.7, 0.3]
)

In [3]:
res = dense_retriever.invoke("how to change Torque Limits ?")

In [4]:
print( res[1].page_content)

These parameters set the torque limits in each operation mode.  
A setting of 100% is equal to the motor rated torque.  
| No.   | Parameter Name                    | Setting Range   | Default   |
|-------|-----------------------------------|-----------------|-----------|
| L7-01 | Forward Torque Limit              | 0 to 300%       | 200%      |
| L7-02 | Reverse Torque Limit              | 0 to 300%       | 200%      |
| L7-03 | Forward Regenerative Torque Limit | 0 to 300%       | 200%      |
| L7-04 | Reverse Regenerative Torque Limit | 0 to 300%       | 200%      |


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
                                PromptTemplate,
                                SystemMessagePromptTemplate,
                                HumanMessagePromptTemplate,
                                ChatPromptTemplate
                                )
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_classic import hub

prompt = """
    You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.
    If you don't know the answer, just say that you don't know.
    Answer in bullet points. Make sure your answer is relevant to the question and it is answered from the context only.
    Question: {question} 
    Context: {context} 
    Answer:
"""

template = ChatPromptTemplate.from_template(prompt)

def format_docs(docs):
    return '\n\n'.join([doc.page_content for doc in docs])

crg_chain = (
    RunnableParallel(
        {
            "context": hybrid_retriever | format_docs,
            "question": RunnablePassthrough()
        }
    )
    | template
    | llm
    | StrOutputParser()
)

In [ ]:
res = crg_chain.invoke("what is Torque Limits and how can i change it?")

print( res )